<a href="https://colab.research.google.com/github/Meenakshi1224-uv/miniproject_etl/blob/main/miniproject_datasetETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**EXTRACT**

In [ ]:
import zipfile

# Path to the ZIP file
zip_file = "/content/online+retail.zip"

# Extract all files
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("online_retail_data")

print("ZIP file extracted successfully!")

ZIP file extracted successfully!


In [ ]:
import os
print(os.listdir("/content/online_retail_data"))

['Online Retail.xlsx']


In [ ]:
import pandas as pd

df = pd.read_excel("/content/online_retail_data/Online Retail.xlsx")
df


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [ ]:
print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB
None


TRANSFORM


In [ ]:
#remove missing customer id
df=df.dropna(subset=["CustomerID"])

In [ ]:
#remove duplicate rows
df=df.drop_duplicates()

In [ ]:
#keep only valid quantity and unitprice
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

In [ ]:
# Convert InvoiceDate to datetime format
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


In [ ]:
# Create TotalPrice column
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

In [ ]:
# Extract Year, Month and Day
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day

print("Transformation completed!")

Transformation completed!


**LOAD**

In [ ]:
import sqlite3

# Connect to SQLite database
conn = sqlite3.connect("online_retail.db")

# Load the cleaned DataFrame into a table named 'retail_sales'
df.to_sql(
    "retail_sales",
    conn,
    if_exists="replace",
    index=False
)

# Close the database connection
conn.close()

print("Data loaded successfully into SQLite!")

Data loaded successfully into SQLite!


In [ ]:
import sqlite3

conn = sqlite3.connect("online_retail.db")
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM retail_sales")
print("Total Rows Loaded:", cursor.fetchone()[0])

conn.close()

Total Rows Loaded: 392692
